In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split

In [58]:
df = pd.read_csv("multi_polynomial_data.csv")
df.sample(5)

,x1,x2,y
299,-2.487915,2.534996,16.849804
372,-2.433342,0.863997,-0.956335
365,-0.683384,0.357842,2.926271
397,-0.426036,2.378015,19.409554
181,-2.319159,-1.452583,10.034365


In [62]:
X = df.iloc[:, 0:2].values
y = df.iloc[:, 2].values

In [60]:
X = X.reshape(-1, 1)
y = y.reshape(len(y), 1)

In [63]:
X.shape, y.shape  

((500, 2), (500,))

In [27]:
def polynomial_features(X, degree):
    X = np.asarray(X)
    n = X.shape[0]
    
    # Create matrix with powers of X
    X_poly = np.ones((n, degree))
    
    for d in range(1, degree + 1):
        X_poly[:, d-1] = X[:, 0] ** d
        
    return X_poly

In [23]:
def poly_features(features, X):
    # create polynomial features using pure numpy, ensure float dtype
    X = np.asarray(X, dtype=float).reshape(-1, 1)
    cols = [(X ** i).flatten() for i in range(1, features + 1)]
    return np.vstack(cols).T


In [ ]:
def split_data(X,y,test_size=0.2,random_state=0):
    np.random.seed(random_state)                  #set the seed for reproducible results
    indices = np.random.permutation(len(X))       #shuffling the indices
    data_test_size = int(X.shape[0] * test_size)  #Get the test size

    train_indices = indices[data_test_size:]
    test_indices = indices[:data_test_size]
    X_train = X[train_indices]
    y_train = y[train_indices]
    X_test = X[test_indices]
    y_test = y[test_indices]
    return X_train, y_train, X_test, y_test

In [26]:
class LinearRegression:
    def __init__(self, learning_rate=0.01, n_iter=1000):
        self.learning_rate = learning_rate
        self.n_iter = n_iter
        self.w = None
        self.b = None
        self.losses = []

    def fit(self, X, y):
        n_samples, n_features = X.shape

        self.w = np.zeros(n_features)
        self.b = 0
        y = np.asarray(y).flatten()

        for _ in range(self.n_iter):
            y_pred = np.dot(X, self.w) + self.b
            error = y_pred - y

            mse = np.mean(error ** 2)
            self.losses.append(mse)

            dw = (2/n_samples) * np.dot(X.T, error)
            db = (2/n_samples) * np.sum(error)

            self.w -= self.learning_rate * dw
            self.b -= self.learning_rate * db

    def predict(self, X):
        return np.dot(X, self.w) + self.b

    def get_losses(self):
        return self.losses

In [78]:
X_train, y_train, X_test, y_test = split_data(X, y, test_size=0.2, random_state=42)

In [34]:
def standardize(X):
    mean = np.mean(X, axis=0)
    std = np.std(X, axis=0)
    return (X - mean) / std

In [ ]:
X_poly = polynomial_features(X_train, degree=2)
X_scaled = standardize(X_poly)
model = LinearRegression(learning_rate=0.001, n_iter=3000)
model.fit(X_scaled, y_train)

In [80]:
print("Learned Weights:", model.w)
print("Learned Bias:", model.b)

Learned Weights: [3.19302188 0.29401085]
Learned Bias: 13.166824074777868


In [ ]:
y_pred = model.predict(X_scaled)

mse = np.mean((y_pred - y_train.flatten()) ** 2)
print("Mean Squared Error:", mse)
r2_score = 1 - (np.sum((y_train.flatten() - y_pred) ** 2) / np.sum((y_train.flatten() - np.mean(y_train.flatten())) ** 2))
print("R² Score:", r2_score)


Mean Squared Error: 67.9978566446174
R² Score: 0.131722714210185
